Question3: Hard (CO4) – Dataset5: Implement a CNN convolution layer using CUDA. (Dataset5: CIFAR-10 Image Dataset)

In [1]:
!pip install numba -q

import numpy as np
import numba
from numba import cuda
import time
import os
import pickle
import tarfile
import urllib.request

# 1. Download and load CIFAR-10
cifar_url = "https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz"
file_name = "cifar-10-python.tar.gz"
extract_folder = "cifar-10-batches-py"

if not os.path.exists(file_name):
    print("Downloading CIFAR-10 dataset (163 MB)...")
    urllib.request.urlretrieve(cifar_url, file_name)
    print("Download complete.")

if not os.path.exists(extract_folder):
    print("Extracting...")
    with tarfile.open(file_name, "r:gz") as tar:
        tar.extractall()
    print("Extraction done.")

# Load first batch (data_batch_1) which contains 10000 images
batch_file = os.path.join(extract_folder, "data_batch_1")
with open(batch_file, "rb") as f:
    batch = pickle.load(f, encoding="bytes")

# Images are stored as flat arrays of 3072 values (32x32 RGB)
# Reshape to (num_images, 3, 32, 32) for easier handling
raw_images = batch[b"data"]          # shape (10000, 3072)
images = raw_images.reshape(-1, 3, 32, 32).astype(np.float32) / 255.0  # normalize [0,1]

# Use only first 8 images for demonstration (to keep runtime short)
num_images = 8
sample_images = images[:num_images]  # shape (8, 3, 32, 32)

Download complete.
Extracting...


/tmp/ipykernel_671/2932511872.py:25: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall()


Extraction done.


In [2]:
# 2. Define the convolution filter

# Simple 3x3 edge detection filter (Sobel vertical)
sobel_x = np.array([[-1, 0, 1],
                    [-2, 0, 2],
                    [-1, 0, 1]], dtype=np.float32)
kernel = sobel_x
K = kernel.shape[0]  # 3
pad = K // 2         # 1

In [3]:
# 3. CUDA convolution kernel

@cuda.jit
def conv2d_gpu(input_imgs, output_imgs, kernel, out_h, out_w, num_imgs, channels):
    """
    Each thread computes one output pixel (y, x) for a given image and channel.
    input_imgs: shape (num_imgs, channels, in_h, in_w)
    output_imgs: shape (num_imgs, channels, out_h, out_w)
    kernel: (K, K) constant
    """
    # Map thread to output pixel
    img = cuda.blockIdx.z        # which image
    ch  = cuda.blockIdx.y        # which channel
    row = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    col = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y  # Wait, careful: we used blockIdx.y for channel, so we need separate indexing.

    # Better: use grid of (out_h, out_w) per image-channel.
    # Let's define one thread per output pixel (row, col) for a specific image and channel.
    # We'll launch blocks over (out_h/block_dim, out_w/block_dim) for each image and channel.
    # Actually, easier: let threads be arranged in 2D grid covering (out_w, out_h) and loop over images/channels.
    # So each thread handles all channels and images? That would be sequential but simpler.
    pass

# I'll rewrite with a cleaner mapping:
@cuda.jit
def conv2d_gpu(input_imgs, output_imgs, kernel, out_h, out_w, num_imgs, channels):
    # Global thread index in 2D grid covers (out_w, out_h)
    col = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    row = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y

    # Loop over all images and channels inside each thread
    for img in range(num_imgs):
        for ch in range(channels):
            if row < out_h and col < out_w:
                acc = 0.0
                # Convolve 3x3 region
                for ki in range(-pad, pad+1):
                    for kj in range(-pad, pad+1):
                        in_row = row + ki
                        in_col = col + kj
                        # Zero-padding: skip if out of bounds
                        if 0 <= in_row < 32 and 0 <= in_col < 32:
                            pix = input_imgs[img, ch, in_row, in_col]
                        else:
                            pix = 0.0
                        acc += pix * kernel[ki+pad, kj+pad]
                output_imgs[img, ch, row, col] = acc

In [4]:
# 4. Prepare data and GPU run

in_h, in_w = 32, 32
out_h = in_h
out_w = in_w
channels = 3

# Output array (same size as input with zero-padding)
output_gpu = np.zeros((num_images, channels, out_h, out_w), dtype=np.float32)

# Move arrays to device
d_input = cuda.to_device(sample_images)
d_output = cuda.device_array_like(output_gpu)
d_kernel = cuda.to_device(kernel)

# Block/grid dimensions: each block 16x16 threads, covering output image
tpb = (16, 16)  # threads per block
blocks_x = (out_w + tpb[0] - 1) // tpb[0]
blocks_y = (out_h + tpb[1] - 1) // tpb[1]
grid = (blocks_x, blocks_y)

print("Running GPU convolution...")
start = time.time()
conv2d_gpu[grid, tpb](d_input, d_output, d_kernel, out_h, out_w, num_images, channels)
cuda.synchronize()
gpu_time = time.time() - start

# Copy result back
output_gpu = d_output.copy_to_host()

Running GPU convolution...


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [5]:
# 5. CPU reference for verification

print("Running CPU convolution for verification...")
output_cpu = np.zeros_like(output_gpu)
start = time.time()
for n in range(num_images):
    for c in range(channels):
        for i in range(out_h):
            for j in range(out_w):
                s = 0.0
                for ki in range(-pad, pad+1):
                    for kj in range(-pad, pad+1):
                        in_i = i + ki
                        in_j = j + kj
                        if 0 <= in_i < 32 and 0 <= in_j < 32:
                            s += sample_images[n, c, in_i, in_j] * kernel[ki+pad, kj+pad]
                output_cpu[n, c, i, j] = s
cpu_time = time.time() - start

# Compare
diff = np.abs(output_gpu - output_cpu)
max_error = np.max(diff)
print(f"\nGPU time: {gpu_time:.4f} sec")
print(f"CPU time: {cpu_time:.4f} sec")
print(f"Max difference between GPU and CPU: {max_error:.2e}")
print("First image, red channel, top-left 3x3 output:")
print(output_gpu[0, 0, :3, :3])

Running CPU convolution for verification...

GPU time: 3.0857 sec
CPU time: 0.4369 sec
Max difference between GPU and CPU: 4.77e-07
First image, red channel, top-left 3x3 output:
[[ 0.3372549  -0.06274509  0.39607847]
 [ 0.23137257  0.07450981  0.7607843 ]
 [ 0.27450982  0.40784314  0.99215686]]
